In [11]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("filtered_data.csv")  # Replace with your file path if needed

# Separate features and target
X = df.drop(columns=["readmitted"])
y = df["readmitted"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Perform KMeans clustering
N_clusters=8
kmeans = KMeans(n_clusters=N_clusters, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Sample 50 of each class (0 and 1) from each cluster
core_set_balanced = []

for cluster_id in range(N_clusters):
    cluster_data = df[df["cluster"] == cluster_id]
    class_0_samples = cluster_data[cluster_data["readmitted"] == 0].sample(n=2500//N_clusters, random_state=42, replace=True)
    class_1_samples = cluster_data[cluster_data["readmitted"] == 1].sample(n=2500//N_clusters, random_state=42, replace=True)
    core_set_balanced.append(pd.concat([class_0_samples, class_1_samples]))

# Combine and shuffle
core_set_final = pd.concat(core_set_balanced).sample(frac=1, random_state=42).reset_index(drop=True)

# Drop the cluster column
core_set_final = core_set_final.drop(columns=["cluster"])

# Save to CSV
core_set_final.to_csv("{} clusters kmeans_balanced_core_set_5000.csv".format(N_clusters), index=False)
